# Notebook 09 — Speciation and Phylogenetics

**Module:** 05 — Biology Fundamentals  
**Notebook:** 09 of 18  
**Prerequisites:** NB08  
**Time estimate:** 75 minutes

> Phylogenetic trees appear in virtually every comparative genomics paper.
> 'Can you read this tree?' is a realistic interview question.

---
## Step 1 — Motivation

Comparative genomics — comparing genomes across species to identify conserved or
rapidly evolving regions — requires knowing the evolutionary relationships between
those species. Every tool that computes conservation scores (PhyloP, phastCons) or
reconstructs ancestral sequences implicitly assumes a phylogenetic tree.

---
## Step 2 — Intuition

A phylogenetic tree is a family tree for species (or genes). Branch length represents
evolutionary time or genetic distance. Branches that diverged recently from a common
ancestor share more DNA than branches that diverged long ago. The root is the common
ancestor of all taxa in the tree.

---
## Step 3 — Biological Background

**Speciation mechanisms:**
- **Allopatric:** geographic isolation separates populations; they diverge independently
- **Sympatric:** speciation within the same geographic area via niche partitioning,
  sexual selection, or polyploidy (common in plants)
- **Reproductive isolation:** the defining criterion — two populations are separate
  species if they cannot interbreed and produce fertile offspring

**Key phylogenetics vocabulary:**
- **Node:** branching point; represents a common ancestor
- **Leaf (tip):** a modern species or sequence
- **Branch length:** amount of evolutionary change along that branch
- **Clade:** a node plus all its descendants (a monophyletic group)
- **Outgroup:** a taxon used to root the tree (more distantly related than all ingroup taxa)
- **Bootstrap value:** % of resampled datasets that recover the same grouping — a
  measure of support; typically >70% is considered well-supported

**Tree-building methods (conceptual):**

| Method | Approach | Speed | Common use |
|--------|----------|-------|------------|
| UPGMA | Distance matrix + average linkage | Fastest | Quick trees (assumes molecular clock) |
| Neighbour-joining | Distance matrix + NJ algorithm | Fast | Medium datasets |
| Maximum likelihood | Statistical model of substitution | Slow | Publication-quality trees |
| Bayesian | MCMC + prior on tree topology | Very slow | Publication + credible intervals |

**Molecular clock hypothesis:** if substitution rate is constant, branch length is
proportional to time. Calibrated against fossils or known divergence events.

---
## Step 4 — Mathematical Explanation

**UPGMA (Unweighted Pair Group Method with Arithmetic Mean):**
1. Start with N leaves, each in its own cluster
2. Find the pair with the smallest distance → merge into a new node
3. Recompute distances to the new node as arithmetic mean of distances to the merged pair
4. Repeat until one node remains (the root)

**Pairwise genetic distance** (Jukes-Cantor correction for multiple substitutions):
$$d_{JC} = -\frac{3}{4} \ln\!\left(1 - \frac{4}{3} p\right)$$

where p = proportion of differing sites. This corrects for the fact that observed
differences undercount true substitutions (some sites mutate multiple times).

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [ ]:
# Cell 6.1 — Jukes-Cantor distance correction
def jukes_cantor_distance(p: float) -> float:
    """Jukes-Cantor distance from observed proportion of differing sites p."""
    if p >= 0.75:
        return np.inf  # no correction possible
    return -0.75 * np.log(1.0 - (4.0/3.0) * p)

def pairwise_distances(sequences: dict) -> dict:
    """Compute all pairwise JC distances from a dict of aligned DNA sequences."""
    names = list(sequences.keys())
    n = len(names)
    dist = {}
    for i in range(n):
        for j in range(i+1, n):
            seq1 = sequences[names[i]]
            seq2 = sequences[names[j]]
            assert len(seq1) == len(seq2), "Sequences must be aligned"
            p = sum(a != b for a, b in zip(seq1, seq2)) / len(seq1)
            d = jukes_cantor_distance(p)
            dist[(names[i], names[j])] = d
            dist[(names[j], names[i])] = d
    for name in names:
        dist[(name, name)] = 0.0
    return dist

# Synthetic aligned sequences for 5 species
sequences = {
    'Human':  'ATGCGATCGATCGATCGATA',
    'Chimp':  'ATGCGATCGATCGATCGATA',  # identical to human
    'Mouse':  'ATGCAATCGATCGATCGATA',  # 1 difference
    'Zebrafish': 'ATGCAATCAATCGATCGTTA', # 3 differences
    'Fruitfly':  'ATGCAATCAATCGATAGTCA', # 5 differences
}

dist = pairwise_distances(sequences)
names = list(sequences.keys())
print("Pairwise distances (Jukes-Cantor corrected):")
header = f"{'':>12}" + ''.join(f"{n[:6]:>10}" for n in names)
print(header)
for n1 in names:
    row = f"{n1[:12]:>12}" + ''.join(f"{dist[(n1,n2)]:>10.4f}" for n2 in names)
    print(row)

In [ ]:
# Cell 6.2 — UPGMA tree construction from scratch
def upgma(dist_matrix: dict, names: list) -> list:
    """
    Build a UPGMA tree from pairwise distances.
    Returns a list of merge events: [(name_A, name_B, distance, new_cluster_name), ...]
    """
    # Working copy: each name is a cluster; cluster size tracks number of leaves
    clusters = {n: 1 for n in names}
    d = {k: v for k, v in dist_matrix.items()}
    merges = []
    cluster_names = list(names)
    node_counter = len(names)

    while len(cluster_names) > 1:
        # Find smallest distance between distinct clusters
        min_d = np.inf
        best = (None, None)
        for i in range(len(cluster_names)):
            for j in range(i+1, len(cluster_names)):
                ci, cj = cluster_names[i], cluster_names[j]
                if d[(ci, cj)] < min_d:
                    min_d = d[(ci, cj)]
                    best = (ci, cj)

        ci, cj = best
        new_name = f"Node{node_counter}"
        node_counter += 1
        merges.append((ci, cj, min_d / 2, new_name))  # branch length = half the merge distance

        # Update distances to new cluster (UPGMA: weighted average)
        ni, nj = clusters[ci], clusters[cj]
        new_cluster_size = ni + nj
        for ck in cluster_names:
            if ck in (ci, cj):
                continue
            new_d = (ni * d[(ci, ck)] + nj * d[(cj, ck)]) / new_cluster_size
            d[(new_name, ck)] = new_d
            d[(ck, new_name)] = new_d
        d[(new_name, new_name)] = 0.0

        clusters[new_name] = new_cluster_size
        cluster_names = [c for c in cluster_names if c not in (ci, cj)] + [new_name]

    return merges

merges = upgma(dist, names)
print("UPGMA merge order (Newick-like):")
for ci, cj, branch_len, new_name in merges:
    print(f"  ({ci}, {cj}) → {new_name}  [branch length = {branch_len:.4f}]")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — JC distance correction curve + simplified tree diagram
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Panel 1: JC distance vs. raw proportion
ax = axes[0]
p_raw = np.linspace(0.001, 0.74, 200)
d_jc = np.array([jukes_cantor_distance(p) for p in p_raw])
ax.plot(p_raw, p_raw, color='gray', ls='--', label='Observed p (no correction)', lw=1.5)
ax.plot(p_raw, d_jc, color='steelblue', lw=2, label='JC corrected distance')
ax.set_xlabel('Observed proportion of differing sites (p)')
ax.set_ylabel('Genetic distance')
ax.set_title('Jukes-Cantor correction:\naccounts for multiple hits at same site')
ax.legend()
ax.axvline(0.75, color='tomato', ls=':', lw=1, label='p=0.75 (saturation)')

# Panel 2: Stylized phylogenetic tree (based on UPGMA output above)
ax = axes[1]
ax.set_xlim(-0.05, 0.25); ax.set_ylim(-0.5, 5.5)
ax.axis('off')
ax.set_title('UPGMA tree: 5-species example', fontsize=11)

# Hardcoded tree from UPGMA output
# Leaf y-positions
leaf_y = {'Human': 4.8, 'Chimp': 3.8, 'Mouse': 2.8, 'Zebrafish': 1.8, 'Fruitfly': 0.8}
leaf_x = 0.20  # all leaves at right

# Draw leaves
for name, y in leaf_y.items():
    ax.text(leaf_x + 0.01, y, name, va='center', fontsize=9)

# Draw simplified dendrogram branches
node_positions = {
    'Human': (leaf_x, leaf_y['Human']),
    'Chimp': (leaf_x, leaf_y['Chimp']),
    'Mouse': (leaf_x, leaf_y['Mouse']),
    'Zebrafish': (leaf_x, leaf_y['Zebrafish']),
    'Fruitfly': (leaf_x, leaf_y['Fruitfly']),
}
# Human+Chimp merge at x=0.19
hc_x = 0.185; hc_y = (leaf_y['Human'] + leaf_y['Chimp']) / 2
ax.plot([leaf_x, hc_x], [leaf_y['Human'], leaf_y['Human']], 'k-', lw=1.5)
ax.plot([leaf_x, hc_x], [leaf_y['Chimp'], leaf_y['Chimp']], 'k-', lw=1.5)
ax.plot([hc_x, hc_x], [leaf_y['Chimp'], leaf_y['Human']], 'k-', lw=1.5)

# (Human,Chimp)+Mouse merge at x=0.16
hcm_x = 0.14; hcm_y = (hc_y + leaf_y['Mouse']) / 2
ax.plot([hc_x, hcm_x], [hc_y, hc_y], 'k-', lw=1.5)
ax.plot([leaf_x, hcm_x], [leaf_y['Mouse'], hcm_y], 'k-', lw=1.5)
ax.plot([hcm_x, hcm_x], [leaf_y['Mouse'], hc_y], 'k-', lw=1.5)

# Zebrafish at x=0.10
zf_x = 0.08; zf_y = (hcm_y + leaf_y['Zebrafish']) / 2
ax.plot([hcm_x, zf_x], [hcm_y, hcm_y], 'k-', lw=1.5)
ax.plot([leaf_x, zf_x], [leaf_y['Zebrafish'], zf_y], 'k-', lw=1.5)
ax.plot([zf_x, zf_x], [leaf_y['Zebrafish'], hcm_y], 'k-', lw=1.5)

# Fruitfly (outgroup) at x=0.03
root_x = 0.02; root_y = (zf_y + leaf_y['Fruitfly']) / 2
ax.plot([zf_x, root_x], [zf_y, zf_y], 'k-', lw=1.5)
ax.plot([leaf_x, root_x], [leaf_y['Fruitfly'], root_y], 'k-', lw=1.5)
ax.plot([root_x, root_x], [leaf_y['Fruitfly'], zf_y], 'k-', lw=1.5)

ax.text(0.01, 2.3, 'Root\n(ancestor)', ha='center', fontsize=8, color='steelblue')

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `jukes_cantor_distance(p)` and `upgma(dist_matrix, names)` from scratch.
   Test on the 5-species distance matrix above.
2. Read a Newick string: `((Human,Chimp),(Mouse,(Zebrafish,Fruitfly)));`
   Draw the tree by hand. Which species is the outgroup? Which pair is most
   closely related?
3. A phylogenetic tree shows bootstrap values of 45 on one branch and 95 on another.
   Which grouping would you trust more? Why?
4. Why does the JC distance correction approach infinity as p → 0.75? What does
   p = 0.75 mean biologically?

---
## Quiz — Active Recall

1. What is a clade? Give an example from the mammal tree.
2. What does branch length represent in a phylogenetic tree?
3. What is an outgroup and why do we need one?
4. What is the molecular clock hypothesis? When does it fail?
5. Why does the JC distance formula correct for multiple hits?

---
## Reflection

**Date completed:** ____________________

> *[A paper uses a maximum-likelihood tree rather than UPGMA. What advantage does
> maximum likelihood offer? When would you use UPGMA instead?]*

---
*Next: `10_cell_signaling_and_gene_regulation.ipynb`*